In [ ]:
import glob
import os
import random
from typing import Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader, Dataset, Subset
from tqdm import tqdm

In [ ]:
DATASET_DIR = "dataset"
IMAGES_DIR = os.path.join(DATASET_DIR, "images")
LABELS_DIR = os.path.join(DATASET_DIR, "labels")
CHECKPOINT_PATH = "qrbitnet.pt"
ONNX_PATH = "qrbitnet.onnx"

QR_SIZE = 21
PATCH_SIZE = 10
IMAGE_SIZE = QR_SIZE * PATCH_SIZE

SPLIT_VALUE = 0.2
BATCH_SIZE = 32
EPOCHS = 50

LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 0
SEED = 42
GRAD_CLIP_NORM = 0.5
USE_AMP = False
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
def seed_everything(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
torch.backends.cudnn.benchmark = True

print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
print("USE_AMP:", USE_AMP)

In [ ]:
class QRCodeDataset(Dataset):
    def __init__(
        self,
        images_dir: str,
        labels_dir: str,
        image_size: int = IMAGE_SIZE,
    ) -> None:
        self.images_dir = images_dir
        self.labels_dir = labels_dir
        self.image_size = image_size

        image_paths = sorted(glob.glob(os.path.join(self.images_dir, "*.png")))
        items = []

        for image_path in image_paths:
            base = os.path.splitext(os.path.basename(image_path))[0]
            label_path = os.path.join(self.labels_dir, f"{base}.txt")

            if os.path.exists(label_path):
                items.append((image_path, label_path))

        if not items:
            raise RuntimeError(
                f"No image/label pairs found in {images_dir} and {labels_dir}"
            )

        self.items = items

    def __len__(self) -> int:
        return len(self.items)

    def read_label(self, path: str) -> torch.Tensor:
        with open(path, "r", encoding="utf-8") as file:
            lines = [line.strip() for line in file.readlines() if line.strip()]

        matrix = []

        for line in lines:
            row = [int(bit) for bit in line.split()]
            matrix.append(row)

        label = torch.tensor(matrix, dtype=torch.float32)

        if label.shape != (QR_SIZE, QR_SIZE):
            raise ValueError(
                f"Bad label shape {tuple(label.shape)} for {path}; "
                f"expected {(QR_SIZE, QR_SIZE)}"
            )

        return label

    def __getitem__(self, index: int) -> Tuple[torch.Tensor, torch.Tensor, str]:
        image_path, label_path = self.items[index]

        image = Image.open(image_path).convert("L")
        array = np.array(image, dtype=np.uint8)

        x = torch.from_numpy(array.astype(np.float32) / 255.0).unsqueeze(0)

        if not torch.isfinite(x).all():
            raise ValueError(f"Non-finite image tensor from {image_path}")

        y = self.read_label(label_path)

        return x, y, os.path.basename(image_path)

In [ ]:
def make_loaders() -> Tuple[DataLoader, DataLoader]:
    base_dataset = QRCodeDataset(IMAGES_DIR, LABELS_DIR)
    n = len(base_dataset)

    generator = torch.Generator().manual_seed(SEED)
    indices = torch.randperm(n, generator=generator).tolist()
    train_size = int((1.0 - SPLIT_VALUE) * n)
    train_indices = indices[:train_size]
    val_indices = indices[train_size:]

    train_dataset = QRCodeDataset(IMAGES_DIR, LABELS_DIR)
    val_dataset = QRCodeDataset(IMAGES_DIR, LABELS_DIR)

    train_set = Subset(train_dataset, train_indices)
    val_set = Subset(val_dataset, val_indices)

    train_loader = DataLoader(
        train_set,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE == "cuda"),
        drop_last=False,
    )
    val_loader = DataLoader(
        val_set,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE == "cuda"),
        drop_last=False,
    )
    return train_loader, val_loader

train_loader, val_loader = make_loaders()

In [ ]:
def group_count(channels: int, preferred: int = 8) -> int:
    for groups in [preferred, 4, 2, 1]:
        if channels % groups == 0:
            return groups
    return 1

class ConvGNAct(nn.Module):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 3,
        stride: int = 1,
        padding: Optional[int] = None,
        dropout: float = 0.0,
    ) -> None:
        super().__init__()
        if padding is None:
            padding = kernel_size // 2

        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, stride=stride, padding=padding, bias=False)
        self.gn = nn.GroupNorm(group_count(out_channels), out_channels)
        self.act = nn.SiLU(inplace=True)
        self.drop = nn.Dropout2d(dropout) if dropout > 0 else nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.drop(self.act(self.gn(self.conv(x))))


class ResBlock(nn.Module):
    def __init__(
        self, 
        channels: int, 
        dropout: float = 0.0, 
        dilation: int = 1
    ) -> None:
        super().__init__()
        padding = dilation

        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=padding, dilation=dilation, bias=False)
        self.gn1 = nn.GroupNorm(group_count(channels), channels)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)
        self.gn2 = nn.GroupNorm(group_count(channels), channels)
        self.act = nn.SiLU(inplace=True)
        self.drop = nn.Dropout2d(dropout) if dropout > 0 else nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        hidden = self.act(self.gn1(self.conv1(x)))
        hidden = self.drop(hidden)
        hidden = self.gn2(self.conv2(hidden))
        return self.act(x + hidden)


class QRBitNet(nn.Module):
    def __init__(
        self,
        base_width: int = 32,
        grid_width: int = 128,
        grid_depth: int = 6,
        dropout: float = 0.04,
    ) -> None:
        super().__init__()
        channels1 = base_width
        channels2 = base_width * 2
        channels3 = base_width * 3
        channels4 = base_width * 4

        self.image_encoder = nn.Sequential(
            ConvGNAct(1, channels1, 3, stride=1, dropout=0.0),
            ResBlock(channels1, dropout=dropout),
            ConvGNAct(channels1, channels2, 3, stride=2, dropout=dropout),
            ResBlock(channels2, dropout=dropout),
            ResBlock(channels2, dropout=dropout, dilation=2),
            ConvGNAct(channels2, channels3, 3, stride=2, dropout=dropout),
            ResBlock(channels3, dropout=dropout),
            ResBlock(channels3, dropout=dropout, dilation=2),
            ConvGNAct(channels3, channels4, 3, stride=2, dropout=dropout),
            ResBlock(channels4, dropout=dropout),
        )
        self.to_grid = nn.AdaptiveAvgPool2d((QR_SIZE, QR_SIZE))

        yy, xx = torch.meshgrid(
            torch.linspace(-1.0, 1.0, QR_SIZE),
            torch.linspace(-1.0, 1.0, QR_SIZE),
            indexing="ij",
        )
        coordinates = torch.stack([yy, xx], dim=0).unsqueeze(0)
        self.register_buffer("coords", coordinates, persistent=False)

        self.grid_projection = ConvGNAct(channels4 + 2, grid_width, 1, stride=1, padding=0, dropout=0.0)
        self.grid_blocks = nn.Sequential(
            *[
                ResBlock(
                    grid_width,
                    dropout=dropout,
                    dilation=2 if i % 3 == 2 else 1,
                )
                for i in range(grid_depth)
            ]
        )
        self.head = nn.Conv2d(grid_width, 1, kernel_size=1, bias=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.image_encoder(x)
        x = torch.nn.functional.interpolate(
            x,
            size=(QR_SIZE, QR_SIZE),
            mode="bilinear",
            align_corners=False,
        )
        coordinates = self.coords.to(device=x.device, dtype=x.dtype).expand(x.size(0), -1, -1, -1)
        x = torch.cat([x, coordinates], dim=1)
        x = self.grid_projection(x)
        x = self.grid_blocks(x)
        x = self.head(x).squeeze(1)
        return x

In [ ]:
@torch.no_grad()
def compute_metrics(logits: torch.Tensor, y_true: torch.Tensor) -> Tuple[float, float]:
    probabilities = torch.sigmoid(logits)
    y_pred = (probabilities > 0.5).to(torch.float32)

    bit_accuracy = (y_pred == y_true).to(torch.float32).mean().item()
    correct_samples = (y_pred == y_true).view(y_true.size(0), -1).all(dim=1)
    exact_accuracy = correct_samples.to(torch.float32).mean().item()

    return bit_accuracy, exact_accuracy


def train_one_epoch(
    model: torch.nn.Module,
    loader: DataLoader,
    optim: torch.optim.Optimizer,
    loss_fn: torch.nn.Module,
    device: str,
    scaler: Optional[torch.amp.GradScaler] = None,
) -> float:
    model.train()
    total_loss = 0.0
    total_seen = 0
    skipped = 0
    progress_bar = tqdm(loader, desc="Train", leave=False)

    for batch_index, (x, y, names) in enumerate(progress_bar):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True).float()

        if not torch.isfinite(x).all() or not torch.isfinite(y).all():
            skipped += x.size(0)
            bad_name = names[0] if len(names) else "unknown"
            print(f"Skipping non-finite input batch {batch_index}, first file: {bad_name}")
            continue

        optim.zero_grad(set_to_none=True)
        use_amp = (scaler is not None) and (device == "cuda") and USE_AMP

        with torch.autocast(device_type=device, dtype=torch.float16, enabled=use_amp):
            logits = model(x)

        logits_for_loss = logits.float()
        loss = loss_fn(logits_for_loss, y)

        if not torch.isfinite(loss):
            skipped += x.size(0)
            bad_name = names[0] if len(names) else "unknown"

            with torch.no_grad():
                min = float(logits_for_loss.detach().min().cpu()) if torch.isfinite(logits_for_loss).any() else float("nan")
                max = float(logits_for_loss.detach().max().cpu()) if torch.isfinite(logits_for_loss).any() else float("nan")

            print(
                f"Skipping non-finite loss at batch {batch_index}, first file: {bad_name}, "
                f"logits range: [{min:.3g}, {max:.3g}]"
            )
            continue

        if use_amp:
            scaler.scale(loss).backward()
            scaler.unscale_(optim)
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)

            if torch.isfinite(grad_norm):
                scaler.step(optim)
                scaler.update()
            else:
                skipped += x.size(0)
                print(f"Skipping optimizer step at batch {batch_index}: non-finite grad norm")
                scaler.update()
                continue
        else:
            loss.backward()
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)

            if torch.isfinite(grad_norm):
                optim.step()
            else:
                skipped += x.size(0)
                print(f"Skipping optimizer step at batch {batch_index}: non-finite grad norm")
                continue

        batch_size = x.size(0)
        total_loss += float(loss.item()) * batch_size
        total_seen += batch_size
        progress_bar.set_postfix(loss=f"{loss.item():.4f}", skipped=skipped)

    if total_seen == 0:
        raise RuntimeError("All training batches were skipped because loss/gradients were non-finite.")

    if skipped:
        print(f"Warning: skipped {skipped} training samples due to non-finite values.")

    return total_loss / total_seen

@torch.no_grad()
def eval_one_epoch(
    model: torch.nn.Module,
    loader: DataLoader,
    loss_fn: torch.nn.Module,
    device: str,
) -> Tuple[float, float, float]:
    model.eval()
    total_loss = 0.0
    total_bits = 0
    correct_bits = 0
    correct_samples = 0
    total_samples = 0
    progress_bar = tqdm(loader, desc="Val", leave=False)

    for x, y, _ in progress_bar:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        logits = model(x).float()
        loss = loss_fn(logits, y.float())
        total_loss += loss.item() * x.size(0)

        probs = torch.sigmoid(logits)
        pred = (probs > 0.5).to(torch.float32)
        correct = (pred == y)
        correct_bits += int(correct.sum().item())
        total_bits += int(correct.numel())
        correct_samples += int(correct.view(y.size(0), -1).all(dim=1).sum().item())
        total_samples += int(y.size(0))

        bit_accuracy = correct_bits / max(total_bits, 1)
        exact_accuracy = correct_samples / max(total_samples, 1)
        progress_bar.set_postfix(bit_accuracy=f"{bit_accuracy:.4f}", exact_accuracy=f"{exact_accuracy:.4f}")

    return (
        total_loss / len(loader.dataset),
        correct_bits / max(total_bits, 1),
        correct_samples / max(total_samples, 1),
    )

In [ ]:
model = QRBitNet().to(DEVICE)

parameters_count = sum(parameter.numel() for parameter in model.parameters())
print(f"Parameters: {parameters_count:,}")

In [ ]:
best_bit_accuracy = -1.0
best_exact_accuracy = -1.0
best_epoch = -1
start_epoch = 1

if os.path.exists(CHECKPOINT_PATH):
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
    model.load_state_dict(checkpoint["model_state"])

    best_bit_accuracy = checkpoint.get("best_bit_accuracy", -1.0)
    best_exact_accuracy = checkpoint.get("best_exact_accuracy", -1.0)
    best_epoch = checkpoint.get("best_epoch", -1)
    start_epoch = checkpoint.get("epoch", best_epoch) + 1
    
    print(
        f"Loaded {CHECKPOINT_PATH} | "
        f"Best bit accuracy: {best_bit_accuracy:.4f} | "
        f"Best exact accuracy: {best_exact_accuracy:.4f} | "
        f"Best epoch: {best_epoch} | "
        f"Resume epoch: {start_epoch}"
    )

In [ ]:
loss_fn = nn.BCEWithLogitsLoss()
optim = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    eps=1e-7,
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optim,
    mode="max",
    factor=0.5,
    patience=5,
    min_lr=2e-6,
)
scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE == "cuda" and USE_AMP))

In [ ]:
for epoch in range(start_epoch, EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, optim, loss_fn, DEVICE, scaler=scaler)
    val_loss, val_bit_accuracy, val_exact_accuracy = eval_one_epoch(model, val_loader, loss_fn, DEVICE)
    scheduler.step(val_exact_accuracy)

    lr = optim.param_groups[0]["lr"]
    print(
        f"Epoch {epoch}/{EPOCHS} | Learning rate: {lr:.2e} | "
        f"Training loss: {train_loss:.4f} | Validation loss: {val_loss:.4f} | "
        f"Validation bit accuracy: {val_bit_accuracy:.4f} | "
        f"Validation exact accuracy: {val_exact_accuracy:.4f}"
    )

    improved = (val_exact_accuracy > best_exact_accuracy) or (
        val_exact_accuracy == best_exact_accuracy and val_bit_accuracy > best_bit_accuracy
    )
    if improved:
        best_bit_accuracy = val_bit_accuracy
        best_exact_accuracy = val_exact_accuracy
        best_epoch = epoch
        torch.save(
            {
                "epoch": epoch,
                "model_state": model.state_dict(),
                "best_bit_accuracy": best_bit_accuracy,
                "best_exact_accuracy": best_exact_accuracy,
                "best_epoch": best_epoch,
                "config": {
                    "QR_SIZE": QR_SIZE,
                    "PATCH_SIZE": PATCH_SIZE,
                    "IMAGE_SIZE": IMAGE_SIZE,
                    "model": "QRBitNet",
                },
            },
            CHECKPOINT_PATH,
        )
        print(f"New best model saved with validation exact accuracy: {best_exact_accuracy:.4f}.")

print(
    f"\nTraining finished. Best bit accuracy: {best_bit_accuracy:.4f}; "
    f"best exact accuracy: {best_exact_accuracy:.4f}; best epoch: {best_epoch}."
)

In [ ]:
class QRBitNetBits(nn.Module):
    def __init__(self, base: nn.Module):
        super().__init__()
        self.base = base

    def forward(self, x):
        logits = self.base(x)

        probabilities = torch.sigmoid(logits)
        bits = (probabilities > 0.5).to(torch.float32)

        return bits

In [ ]:
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint["model_state"])
model.eval()

export_model = QRBitNetBits(model).eval()
dummy = torch.randn(1, 1, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE)
torch.onnx.export(
    export_model,
    dummy,
    ONNX_PATH,
    opset_version=17,
    input_names=["image"],
    output_names=["bits"],
    dynamic_axes={
        "image": {0: "batch"},
        "bits": {0: "batch"},
    },
    dynamo=False,
)